# Parsing Allen Institute Taxonomy (AIT) `.h5ad` files

This notebook walks through parsing AIT taxonomy files (BICAN / HMBA basal-ganglia,
produced with the [`scrattch`](https://alleninstitute.github.io/scrattch/) toolkit)
step by step. It mirrors `bkbit/data_translators/ait_taxonomy_parser.py`.

**Key idea:** an `.h5ad` file is an HDF5 container. The taxonomy lives in the tiny
`uns` group; the expression matrix `X` is what makes the files huge (30–105 GB).
We read **only** the taxonomy groups — over HTTP range requests — so nothing is
downloaded in full and `X` is never loaded.

## 1. Install dependencies
`anndata` + `h5py` to read HDF5, `fsspec` + `aiohttp` for lazy remote reads.

In [1]:
%pip install anndata h5py fsspec aiohttp pandas


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. Imports and file URLs

In [2]:
import fsspec
import h5py
import pandas as pd
from anndata.io import read_elem

BASE = (
    "https://released-taxonomies-802451596237-us-west-2.s3.us-west-2.amazonaws.com"
    "/HMBA/BasalGanglia/BICAN_05072025_pre-print_release"
)
URLS = {
    "Human": f"{BASE}/Human_HMBA_basalganglia_AIT_pre-print.h5ad",
    "Macaque": f"{BASE}/Macaque_HMBA_basalganglia_AIT_pre-print.h5ad",
    "Marmoset": f"{BASE}/Marmoset_HMBA_basalganglia_AIT_pre-print.h5ad",
}

# Pick one to explore. Marmoset is the smallest (~30 GB); we still never download it.
SPECIES = "Marmoset"
url = URLS[SPECIES]
url

'https://released-taxonomies-802451596237-us-west-2.s3.us-west-2.amazonaws.com/HMBA/BasalGanglia/BICAN_05072025_pre-print_release/Marmoset_HMBA_basalganglia_AIT_pre-print.h5ad'

## 3. Open the remote file lazily (no download)
`fsspec` opens the URL as a file-like object backed by HTTP range requests. `h5py`
reads only the HDF5 index; group data is fetched on demand when accessed.

In [3]:
fileobj = fsspec.open(url, block_size=8 * 1024 * 1024).open()
h5 = h5py.File(fileobj, "r")

# Top-level groups. X / layers / raw are the big expression matrices — we ignore them.
list(h5.keys())

['X', 'layers', 'obs', 'obsm', 'obsp', 'raw', 'uns', 'var', 'varm', 'varp']

## 4. Explore the `uns` group (where the taxonomy lives)

In [4]:
for k in h5["uns"].keys():
    obj = h5["uns"][k]
    if isinstance(obj, h5py.Group):
        print(f"{k}/  (group) -> {list(obj.keys())[:8]}")
    else:
        print(f"{k}: shape={obj.shape} dtype={obj.dtype}")

batch_condition: shape=(1,) dtype=object
cluster_algorithm: shape=() dtype=object
cluster_info/  (group) -> ['ATAC_index', 'ATAC_lib_quantification_ng', 'ATAC_library_creation_date', 'ATAC_library_name', 'ATAC_library_num_cycles', 'ATAC_library_pool_name', 'ATAC_library_prep_set', 'ATAC_tapestation_avg_size_bp']
dataset_purl: shape=() dtype=object
default_embedding: shape=() dtype=object
filter/  (group) -> ['standard']
gene_annotation_version: shape=() dtype=object
hierarchy/  (group) -> ['Class', 'Group', 'Neighborhood', 'Subclass', 'cluster_id']
hvg/  (group) -> ['flavor']
log1p/  (group) -> []
mode: shape=() dtype=object
neighbors/  (group) -> ['connectivities_key', 'distances_key', 'params']
reference_genome: shape=() dtype=object
schema_version: shape=() dtype=object
title: shape=() dtype=object


### 4a. Taxonomy levels from `uns/hierarchy`
A dict of `level -> position`; sorting by position gives root → leaf order.

In [5]:
hierarchy = read_elem(h5["uns"]["hierarchy"])
levels = [name for name, _ in sorted(hierarchy.items(), key=lambda kv: int(kv[1]))]
print("hierarchy:", hierarchy)
print("levels (root -> leaf):", levels)

hierarchy: {'Class': 1, 'Group': 3, 'Neighborhood': 0, 'Subclass': 2, 'cluster_id': 4}
levels (root -> leaf): ['Neighborhood', 'Class', 'Subclass', 'Group', 'cluster_id']


### 4b. The taxonomy table from `uns/cluster_info`
One row per leaf cluster, with the full ancestor path plus per-level accessions,
CL ontology IDs, and colors. `read_elem` decodes categoricals back to strings and
sets the file's `_index` (`cell_label`) as the DataFrame index.

In [6]:
cluster_info = read_elem(h5["uns"]["cluster_info"])
print("shape:", cluster_info.shape, "| index name:", cluster_info.index.name)
cluster_info[["Neighborhood", "Class", "Subclass", "Group", "Cluster",
              "cluster_id", "accession_group", "CL:ID_group", "color_hex_group"]].head()

shape: (594, 80) | index name: cell_label


,Neighborhood,Class,Subclass,Group,Cluster,cluster_id,accession_group,CL:ID_group,color_hex_group
cell_label,,,,,,,,,
GTTCCTGGTTAGCATG-P0005_1,Nonneuron,OPC-Oligo,Oligodendrocyte,Oligo OPALIN,Marmoset-1,Marmoset-1,CS20250428_GROUP_0025,CL:0000128,#488edc
GGTTACCCACCACAAC-P0005_1,Subpallium GABA,CN LGE GABA,STR D2 MSN,STRd D2 Matrix MSN,Marmoset-82,Marmoset-82,CS20250428_GROUP_0052,CL:4030047,#aec7e8
CCATATTTCTGCAACG-P0005_1,Subpallium GABA,CN LGE GABA,STR D1 MSN,STRd D1 Matrix MSN,Marmoset-83,Marmoset-83,CS20250428_GROUP_0049,CL:4030043,#1f77b4
ATTGACTCAACTAGAA-P0005_1,Nonneuron,OPC-Oligo,Oligodendrocyte,ImOligo,Marmoset-96,Marmoset-96,CS20250428_GROUP_0018,CL:4042031,#711c24
GGTCCATCATAATTGC-P0005_1,Subpallium GABA,CN LGE GABA,STR D2 MSN,STRd D2 Striosome MSN,Marmoset-60,Marmoset-60,CS20250428_GROUP_0054,CL:4030049,#ff9896


> **Note:** `cell_label` (the index) is a *representative cell barcode* per cluster,
> not the cluster ID. Use `cluster_id` / `Cluster` to identify a cluster.

### 4b-i. Checking what `_index` is set to
`_index` is an HDF5 attribute on a dataframe group that records which stored
column becomes the pandas index on read. You can inspect it directly on the raw
group, or read it off the loaded DataFrame's `index.name`. This works on any
dataframe group (`obs`, `var`, `uns/cluster_info`, ...).

In [7]:
grp = h5["uns"]["cluster_info"]  # any dataframe group: h5["obs"], h5["var"], ...

# Method 1: read the raw HDF5 attribute (often bytes -> decode).
idx = grp.attrs["_index"]
idx = idx.decode() if isinstance(idx, bytes) else idx
print("_index attribute:", idx)

# See it alongside the data columns and encoding type:
print("all attrs:", dict(grp.attrs))

# Method 2: on the already-loaded DataFrame, it's the index name.
print("cluster_info.index.name:", cluster_info.index.name)

_index attribute: cell_label
all attrs: {'_index': 'cell_label', 'column-order': array(['Neighborhood', 'Class', 'Subclass', 'Group', 'Cluster',
       'cluster_id', 'cell_type_ontology_term', 'load_id', 'donor_id',
       'assay', 'assay_ontology_term_id', 'organism',
       'organism_ontology_term_id', 'development_stage',
       'anatomical_region', 'anatomical_region_merged',
       'anatomical_region_ontology_term_id',
       'brain_region_ontology_term_id', 'self_reported_sex',
       'self_reported_sex_ontology_term_id', 'self_reported_ethnicity',
       'self_reported_ethnicity_ontology_term_id', 'disease',
       'disease_ontology_term_id', 'suspension_type', 'is_primary_data',
       'RNA_library_creation_date', 'RNA_library_prep_set',
       'RNA_library_name', 'RNA_tapestation_avg_size_bp',
       'RNA_library_num_cycles', 'RNA_lib_quantification_ng',
       'RNA_library_pool_name', 'ATAC_library_creation_date',
       'ATAC_library_prep_set', 'ATAC_library_name',
       'A

### 4c. Node counts per level

In [8]:
for lvl in levels:
    col = "Cluster" if lvl == "cluster_id" else lvl
    print(f"{lvl:<14} {cluster_info[col].nunique()} nodes")

Neighborhood   4 nodes
Class          12 nodes
Subclass       34 nodes
Group          56 nodes
cluster_id     594 nodes


## 5. Same thing via the `AITTaxonomy` helper
The steps above are packaged in `bkbit/data_translators/ait_taxonomy_parser.py`.
Run this from the repo root so the import resolves.

In [9]:
from bkbit.data_translators.ait_taxonomy_parser import AITTaxonomy

# load_obs=False skips the large per-cell table — recommended for remote files.
tax = AITTaxonomy.from_file(url, load_obs=False)
print(tax.summary())

title:           Marmoset_HMBA_basalganglia_consensus_AIT
schema_version:  v1.0
reference_genome:GRCh38
levels:          Neighborhood > Class > Subclass > Group > cluster_id
leaf clusters:   594
  Neighborhood   4 nodes
  Class          12 nodes
  Subclass       34 nodes
  Group          56 nodes
  cluster_id     594 nodes
genes (var):     35,787


In [10]:
# Parent -> child tree edges derived from each cluster's ancestor path.
edges = tax.edges()
print(f"{len(edges)} edges; first 6:")
for e in edges[:6]:
    print("  ", e[0], e[1], "->", e[2], e[3])

696 edges; first 6:
   Neighborhood Nonneuron -> Class OPC-Oligo
   Class OPC-Oligo -> Subclass Oligodendrocyte
   Subclass Oligodendrocyte -> Group Oligo OPALIN
   Group Oligo OPALIN -> cluster_id Marmoset-1
   Neighborhood Subpallium GABA -> Class CN LGE GABA
   Class CN LGE GABA -> Subclass STR D2 MSN


import os

OUT_DIR = "bkbit/data/ait_output"  # relative to the repo root
os.makedirs(OUT_DIR, exist_ok=True)
csv_path = f"{OUT_DIR}/{SPECIES}_taxonomy.csv"
pkl_path = f"{OUT_DIR}/{SPECIES}_cluster_info.pkl"

tax.to_csv(csv_path)
tax.cluster_info.to_pickle(pkl_path)
print("wrote:", csv_path, "and", pkl_path)

In [11]:
import os

os.makedirs("ait_output", exist_ok=True)
csv_path = f"ait_output/{SPECIES}_taxonomy.csv"
pkl_path = f"ait_output/{SPECIES}_cluster_info.pkl"

tax.to_csv(csv_path)
tax.cluster_info.to_pickle(pkl_path)
print("wrote:", csv_path, "and", pkl_path)

wrote: ait_output/Marmoset_taxonomy.csv and ait_output/Marmoset_cluster_info.pkl


os.makedirs("bkbit/data/ait_output", exist_ok=True)
for sp, u in URLS.items():
    t = AITTaxonomy.from_file(u, load_obs=False)
    t.to_csv(f"bkbit/data/ait_output/{sp}_taxonomy.csv")
    t.cluster_info.to_pickle(f"bkbit/data/ait_output/{sp}_cluster_info.pkl")
    print(f"{sp}: {t.cluster_info.shape[0]} clusters -> saved")

In [12]:
for sp, u in URLS.items():
    t = AITTaxonomy.from_file(u, load_obs=False)
    t.to_csv(f"ait_output/{sp}_taxonomy.csv")
    t.cluster_info.to_pickle(f"ait_output/{sp}_cluster_info.pkl")
    print(f"{sp}: {t.cluster_info.shape[0]} clusters -> saved")

Human: 453 clusters -> saved
Macaque: 388 clusters -> saved
Marmoset: 594 clusters -> saved
